# Class Imbalance

## Learning Objectives
1. Implement SMOTE from scratch to understand synthetic over-sampling.
2. Compare four imbalance-handling strategies using sklearn metrics and ROC curves.
3. Apply focal loss in PyTorch for extreme imbalance (fraud detection 1:1000).
4. Tune the decision threshold using a precision-recall curve to maximise F1.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, average_precision_score
from sklearn.datasets import make_classification
from sklearn.utils import resample

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: SMOTE from Scratch (NumPy)

In [ ]:
# SMOTE: for each minority sample, pick k nearest neighbours and interpolate linearly.

def smote_scratch(X_min, n_synthetic, k=5):
    # Generate synthetic minority samples via linear interpolation between neighbours.
    # Args:
    # X_min: minority class samples (n, d)
    # n_synthetic: number of new samples to create
    # k: number of nearest neighbours to interpolate between
    n, d = X_min.shape
    synthetic = np.zeros((n_synthetic, d))
    for i in range(n_synthetic):
        idx = np.random.randint(0, n)
        sample = X_min[idx]
        dists = np.linalg.norm(X_min - sample, axis=1)
        dists[idx] = np.inf
        nn_indices = np.argsort(dists)[:k]
        nn = X_min[np.random.choice(nn_indices)]
        lam = np.random.uniform(0, 1)
        synthetic[i] = sample + lam * (nn - sample)
    return synthetic

# Create imbalanced dataset: 900 majority, 90 minority
X_maj = np.random.randn(900, 2) + [2, 2]
X_min = np.random.randn(90, 2) + [-2, -2]
print(f"Before SMOTE -- Majority: {len(X_maj)}, Minority: {len(X_min)}")

synthetic = smote_scratch(X_min, n_synthetic=810, k=5)
X_min_aug = np.vstack([X_min, synthetic])
print(f"After  SMOTE -- Majority: {len(X_maj)}, Minority (augmented): {len(X_min_aug)}")
print(f"Synthetic samples mean: {synthetic.mean(axis=0).round(3)}")
print(f"Original minority mean: {X_min.mean(axis=0).round(3)}  (should be similar)")
print("SMOTE implementation verified.")


## Level 2: Four Imbalance Strategies + ROC (sklearn)

In [ ]:
# Compare: baseline, class_weight=balanced, under-sampling, over-sampling (SMOTE approx)

X_imb, y_imb = make_classification(
    n_samples=2200, n_features=10, n_informative=5,
    weights=[0.9, 0.1], random_state=42
)
X_train, y_train = X_imb[:1800], y_imb[:1800]
X_test,  y_test  = X_imb[1800:], y_imb[1800:]

strategies_results = {}

# 1. Baseline (no adjustment)
clf_base = LogisticRegression(max_iter=500, random_state=42)
clf_base.fit(X_train, y_train)
strategies_results["Baseline"] = {
    "auc": roc_auc_score(y_test, clf_base.predict_proba(X_test)[:,1]),
    "f1":  f1_score(y_test, clf_base.predict(X_test))
}

# 2. Class weights -- tell the model minority errors cost more
clf_w = LogisticRegression(class_weight="balanced", max_iter=500, random_state=42)
try:
    clf_w.fit(X_train, y_train)
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print("OOM -- reduce dataset size")
        torch.cuda.empty_cache()
    raise
strategies_results["ClassWeight"] = {
    "auc": roc_auc_score(y_test, clf_w.predict_proba(X_test)[:,1]),
    "f1":  f1_score(y_test, clf_w.predict(X_test))
}

# 3. Under-sample majority
n_min = (y_train == 1).sum()
X_us = np.vstack([X_train[y_train==0][:n_min], X_train[y_train==1]])
y_us = np.hstack([np.zeros(n_min), np.ones(n_min)])
clf_us = LogisticRegression(max_iter=500, random_state=42)
clf_us.fit(X_us, y_us)
strategies_results["UnderSample"] = {
    "auc": roc_auc_score(y_test, clf_us.predict_proba(X_test)[:,1]),
    "f1":  f1_score(y_test, clf_us.predict(X_test))
}

# 4. Over-sample minority using sklearn resample (SMOTE approximation)
n_maj = (y_train == 0).sum()
X_os = resample(X_train[y_train==1], n_samples=n_maj, random_state=42)
X_ov = np.vstack([X_train[y_train==0], X_os])
y_ov = np.hstack([np.zeros(n_maj), np.ones(n_maj)])
clf_ov = LogisticRegression(max_iter=500, random_state=42)
clf_ov.fit(X_ov, y_ov)
strategies_results["OverSample"] = {
    "auc": roc_auc_score(y_test, clf_ov.predict_proba(X_test)[:,1]),
    "f1":  f1_score(y_test, clf_ov.predict(X_test))
}

print(f"{'Strategy':<14}  {'ROC-AUC':>8}  {'F1':>6}")
for name, m in strategies_results.items():
    print(f"{name:<14}  {m['auc']:>8.4f}  {m['f1']:>6.4f}")


## Real-World Example 1: Fraud Detection at 1:1000 Imbalance

In [ ]:
# Simulate credit-card fraud: 0.1% positive rate.
# Show that accuracy is misleading; use precision-recall AUC instead.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_fraud, y_fraud = make_classification(
    n_samples=10000, n_features=20, n_informative=8,
    weights=[0.999, 0.001], flip_y=0, random_state=42
)
X_fr_tr, y_fr_tr = X_fraud[:8000], y_fraud[:8000]
X_fr_te, y_fr_te = X_fraud[8000:], y_fraud[8000:]

print(f"Train: {(y_fr_tr==1).sum()} fraud / {len(y_fr_tr)} total")
print(f"Test:  {(y_fr_te==1).sum()} fraud / {len(y_fr_te)} total")

# Naive: predict all 0 gets ~99.9% accuracy -- useless metric for fraud
naive_acc = (y_fr_te == 0).mean()
print(f"Naive accuracy (all-zero predictor): {naive_acc:.4f}  -- misleading!")

clf_fr = RandomForestClassifier(
    n_estimators=50, class_weight="balanced", max_depth=8, random_state=42
)
clf_fr.fit(X_fr_tr, y_fr_tr)
proba_fr = clf_fr.predict_proba(X_fr_te)[:, 1]
pr_auc = average_precision_score(y_fr_te, proba_fr)
print(f"RF balanced -- PR-AUC: {pr_auc:.4f}  (meaningful metric for fraud detection)")
print(classification_report(y_fr_te, (proba_fr > 0.3).astype(int),
                             target_names=["legit", "fraud"], zero_division=0))


## Real-World Example 2: Focal Loss in PyTorch

In [ ]:
# Focal loss down-weights easy negatives, focusing training on hard examples.
# Critical when 99%+ of samples are easy negatives (fraud, object detection).

class FocalLoss(nn.Module):
    # Focal loss: FL(p_t) = -alpha * (1-p_t)^gamma * log(p_t).

    def __init__(self, gamma=2.0, alpha=0.25, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets.float(), reduction="none"
        )
        p_t = torch.exp(-bce)
        focal_weight = self.alpha * (1 - p_t) ** self.gamma
        loss = focal_weight * bce
        return loss.mean() if self.reduction == "mean" else loss.sum()

# 10100 samples: 100 positives (1% imbalance)
X_fl = torch.randn(10100, 16).to(device)
y_fl = torch.zeros(10100).to(device)
y_fl[:100] = 1.0

mdl_fl = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
opt_fl = torch.optim.Adam(mdl_fl.parameters(), lr=1e-3)
focal  = FocalLoss(gamma=2.0, alpha=0.25)
ds_fl  = TensorDataset(X_fl, y_fl)
ld_fl  = DataLoader(ds_fl, batch_size=128, shuffle=True)

for epoch in range(40):
    mdl_fl.train()
    for xb, yb in ld_fl:
        opt_fl.zero_grad()
        focal(mdl_fl(xb).squeeze(1), yb).backward()
        opt_fl.step()

mdl_fl.eval()
with torch.no_grad():
    logits_all = mdl_fl(X_fl).squeeze(1)
    preds = (torch.sigmoid(logits_all) > 0.3).float()
    tp = ((preds == 1) & (y_fl == 1)).sum().item()
    fp = ((preds == 1) & (y_fl == 0)).sum().item()
    fn = ((preds == 0) & (y_fl == 1)).sum().item()
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
print(f"Focal Loss -- Precision: {prec:.3f}  Recall: {rec:.3f}  TP={tp}  FP={fp}")


## Real-World Example 3: Threshold Tuning with PR Curve

In [ ]:
# Default threshold 0.5 is often wrong for imbalanced tasks.
# Use the precision-recall curve to find the threshold that maximises F1.

X_thr, y_thr = make_classification(
    n_samples=5000, n_features=10, n_informative=6,
    weights=[0.9, 0.1], random_state=7
)
clf_thr = LogisticRegression(class_weight="balanced", max_iter=500, random_state=7)
clf_thr.fit(X_thr[:4000], y_thr[:4000])
proba_thr = clf_thr.predict_proba(X_thr[4000:])[:, 1]
y_te_thr  = y_thr[4000:]

precision_arr, recall_arr, thresholds = precision_recall_curve(y_te_thr, proba_thr)
# Compute F1 at each threshold (arrays are len(thresholds)+1 -- trim last)
f1_arr = np.where((precision_arr[:-1] + recall_arr[:-1]) > 0,
                  2 * precision_arr[:-1] * recall_arr[:-1] /
                  (precision_arr[:-1] + recall_arr[:-1]), 0.0)
best_idx = np.argmax(f1_arr)
best_thr  = thresholds[best_idx]

print(f"Default threshold (0.5) F1: {f1_score(y_te_thr, proba_thr >= 0.5):.4f}")
print(f"Optimal threshold         : {best_thr:.4f}")
print(f"Optimal F1                : {f1_arr[best_idx]:.4f}")
print(f"Precision at optimal thr  : {precision_arr[best_idx]:.4f}")
print(f"Recall    at optimal thr  : {recall_arr[best_idx]:.4f}")
print("Rule: tune threshold on a validation set, not the final test set.")
